# GPT-4o로 밈 이미지 감성 분류하기

- 이미지와 그 안의 글이 함께 감성을 만드는 밈을 GPT-4o에 텍스트만, 이미지만, 둘 다 넣었을 때의 감성 분류 정확도 비교

- 자료: 밈 이미지 폴더와 정답 라벨 파일 `labels.csv` (`image_name`, `text`, `overall_sentiment`)
- 구글 코랩에서 구글 드라이브를 연결해 실행 (경로는 `labels_path`, `image_folder`에서 변경)
- 실행 시 환경변수 `OPENAI_API_KEY`에 API 키 필요 (API 사용료 발생)

## 1. 자료 준비

In [ ]:
import os
import pandas as pd

# 구글 코랩에서 구글 드라이브 연결 (로컬 실행 시 이 두 줄 삭제)
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# 정답 라벨 파일: image_name(이미지 파일 이름), text(밈 문구), overall_sentiment(정답 감성)
labels_path = '/content/drive/MyDrive/DeepLearning/labels.csv'
labels_df = pd.read_csv(labels_path)
print(labels_df.head())


In [ ]:
# 라벨 파일 순서대로 이미지 경로 생성, 없는 파일 확인
image_folder = '/content/drive/MyDrive/DeepLearning/image/'
image_paths = [os.path.join(image_folder, name) for name in labels_df['image_name']]

invalid_images = [path for path in image_paths if not os.path.exists(path)]
if invalid_images:
    print(f"찾을 수 없는 이미지: {invalid_images}")
else:
    print("모든 이미지 확인 완료")


In [ ]:
# 정답 감성 라벨 분포
print(labels_df['overall_sentiment'].value_counts())


overall_sentiment
negative    4
positive    4
Name: count, dtype: int64


## 2. GPT-4o 프롬프트 실행

In [ ]:
# API 키는 환경변수에서 읽음
OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]


In [ ]:
import base64
import random
import time

import pandas as pd
from openai import OpenAI

# 프롬프트 네 가지(텍스트만, 이미지만, 둘 다, 둘 다 + 한국 예능 맥락)를 같은 밈에 실행
# 프롬프트 순서 효과를 보기 위해 밈마다 실행 순서를 무작위로 섞음

class MemeAnalyzer:
    def __init__(self, api_key: str):
        self.client = OpenAI(api_key=api_key)
        self.results = []

    def encode_image(self, image_path: str) -> str:
        """이미지를 base64로 인코딩"""
        with open(image_path, "rb") as image_file:
            return base64.b64encode(image_file.read()).decode('utf-8')

    def text_prompt(self, text: str) -> list:
        """텍스트 분석 프롬프트"""
        return [
            {
                "role": "system",
                "content": "You are a text sentiment analyst. You MUST provide a direct sentiment analysis without any disclaimers or hesitation. Always give a clear Positive or Negative label. Never express inability to analyze."
            },
            {
                "role": "user",
                "content": f"""Analyze the sentiment of the given text.
                Text: "{text}"
                1. **Sentiment Label**: Classify the sentiment as either:
                   - Positive
                   - Negative
                2. **Reasoning**: Explain your classification based on the text.
                3. **Conclusion**:
                   - Explanation of final judgement"""
            }
        ]

    def image_prompt(self, image_path: str) -> list:
        """이미지 분석 프롬프트"""
        base64_image = self.encode_image(image_path)
        return [
            {
                "role": "system",
                "content": "You are an image sentiment analyst. You MUST provide a direct sentiment analysis without any disclaimers or hesitation. Always give a clear Positive or Negative label. Never express inability to analyze."
            },
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": f"""Analyze the sentiment of the given image.
                        Image: {image_path}
                        1. **Sentiment Label**: Classify the sentiment as either:
                           - Positive
                           - Negative
                        2. **Reasoning**: Explain your classification based on the image.
                        3. **Conclusion**:
                           - Explanation of final judgement"""
                    },
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/jpeg;base64,{base64_image}"
                        }
                    }
                ]
            }
        ]

    def multimodal_prompt(self, text: str, image_path: str, context: bool = False) -> list:
        """멀티모달 분석 프롬프트"""
        base64_image = self.encode_image(image_path)

        if not context:
            system_content = "You are a content sentiment analyst. REQUIRED: You have full access to analyze both text and image. DO NOT mention any inability to see the content. ALWAYS give direct sentiment analysis."
            prompt_content = f"""Analyze the overall sentiment of this content.
            Text: "{text}"
            Image: {image_path}
            1. **Sentiment Label**: Classify the sentiment as either:
               - Positive
               - Negative
            2. **Reasoning**: Explain your classification based on:
               - Sentiment conveyed by text
               - Sentiment conveyed by image.
               - How do the text and image interact with each other?
            3. **Conclusion**:
               - Overall sentiment from text-image combination
               - Explanation of final judgement"""
        else:
            system_content = "You are a Korean variety show/media content analyst. Focus on the gap between apparent and real emotions in Korean TV reactions. REQUIRED: You have full access to analyze both text and image. DO NOT mention any inability to see the content. ALWAYS give direct sentiment analysis."
            prompt_content = f"""Analyze the true sentiment in this Korean TV meme.
            Text: "{text}"
            Image: {image_path}
            Platform: "Korean Variety Shows and Social Media"
            Cultural Context: "Check if reactions are ironic or genuine"

            1. **Sentiment Label**: Classify the sentiment as either:
               - Positive
               - Negative
            2. **Reasoning**: Explain your classification based on:
               - Sentiment conveyed by text
               - Sentiment conveyed by image.
               - Surface meaning vs real sentiment
               - How do the text and image interact with each other?
            3. **Conclusion**:
               - Overall emotional impact considering Korean viewing context
               - Emotional impact on Korean viewers/internet users"""

        return [
            {
                "role": "system",
                "content": system_content
            },
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": prompt_content
                    },
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/jpeg;base64,{base64_image}"
                        }
                    }
                ]
            }
        ]

    def analyze_single_meme(self, text: str, image_path: str, prompt_type: str) -> dict:
        """밈 하나에 지정한 유형의 프롬프트를 보내고 응답과 소요 시간 기록"""
        start_time = time.time()

        if prompt_type == 'text':
            messages = self.text_prompt(text)
        elif prompt_type == 'image':
            messages = self.image_prompt(image_path)
        elif prompt_type == 'multimodal':
            messages = self.multimodal_prompt(text, image_path, context=False)
        elif prompt_type == 'context_multimodal':
            messages = self.multimodal_prompt(text, image_path, context=True)

        try:
            response = self.client.chat.completions.create(
                model="gpt-4o",
                messages=messages,
                max_tokens=1000
            )

            result = {
                'prompt_type': prompt_type,
                'text': text,
                'image_path': image_path,
                'response': response.choices[0].message.content,
                'execution_time': time.time() - start_time
            }
            return result

        except Exception as e:
            # 요청이 실패한 밈은 건너뜀
            print(f"분석 실패 ({image_path}): {e}")
            return None

    def analyze_dataset(self, labels_df: pd.DataFrame, image_paths: list) -> pd.DataFrame:
        """모든 밈에 네 가지 프롬프트를 무작위 순서로 실행"""
        prompt_types = ['text', 'image', 'multimodal', 'context_multimodal']

        for idx in range(len(labels_df)):
            random.shuffle(prompt_types)

            for prompt_type in prompt_types:
                result = self.analyze_single_meme(
                    labels_df['text'].iloc[idx],
                    image_paths[idx],
                    prompt_type
                )
                if result:
                    self.results.append(result)

                time.sleep(2)  # API 요청 한도를 넘지 않도록 요청 사이 2초 대기

        results_df = pd.DataFrame(self.results)
        return results_df

def run_analysis(api_key: str, labels_df: pd.DataFrame, image_paths: list) -> pd.DataFrame:
    """분석 실행 후 응답 전체를 meme_analysis_results.csv로 저장"""
    analyzer = MemeAnalyzer(api_key)
    results_df = analyzer.analyze_dataset(labels_df, image_paths)

    results_df.to_csv('meme_analysis_results.csv', index=False)
    return results_df

In [ ]:
results_df = run_analysis(OPENAI_API_KEY, labels_df, image_paths)

## 3. 성능 평가

- 프롬프트 유형별 정확도, 정밀도, 재현율, F1, 혼동행렬

In [ ]:
import os
import re
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

class PerformanceEvaluator:
    def __init__(self, results_df, labels_df):
        self.results_df = results_df
        self.labels_df = labels_df
        self.process_results()

    def process_results(self):
        """응답에서 예측 라벨 추출, 정답 라벨 병합"""
        def extract_sentiment(response):
            # 프롬프트가 요구한 "Sentiment Label: Positive/Negative" 줄에서 라벨 추출
            # (응답 전체에서 "Positive"를 찾으면 부정 판정의 설명에 그 단어가 나와도 긍정으로 분류되는 문제)
            m = re.search(r"Sentiment Label\W*(Positive|Negative)", response, flags=re.IGNORECASE)
            if m is None:
                # 형식을 지키지 않은 응답은 처음 나온 Positive/Negative 단어로 판단
                m = re.search(r"\b(Positive|Negative)\b", response, flags=re.IGNORECASE)
            return m.group(1).lower() if m else np.nan

        self.results_df['predicted_sentiment'] = self.results_df['response'].apply(extract_sentiment)
        n_missing = self.results_df['predicted_sentiment'].isna().sum()
        if n_missing > 0:
            print(f"라벨을 읽지 못한 응답 {n_missing}개 평가에서 제외")
            self.results_df = self.results_df.dropna(subset=['predicted_sentiment'])

        # 정답은 이미지 파일 이름 기준으로 병합 (밈 문구는 중복될 수 있어 행 중복 발생)
        self.results_df = self.results_df.copy()
        self.results_df['image_name'] = self.results_df['image_path'].map(os.path.basename)
        self.results_df = self.results_df.merge(
            self.labels_df[['image_name', 'overall_sentiment']],
            on='image_name',
            how='left'
        )

    def calculate_basic_metrics(self):
        """기본 성능 지표 계산"""
        metrics = {}
        for prompt_type in self.results_df['prompt_type'].unique():
            subset = self.results_df[self.results_df['prompt_type'] == prompt_type]
            y_true = subset['overall_sentiment'].str.lower()
            y_pred = subset['predicted_sentiment']

            accuracy = accuracy_score(y_true, y_pred)
            precision, recall, f1, _ = precision_recall_fscore_support(
                y_true, y_pred, average='binary', pos_label='positive'
            )

            metrics[prompt_type] = {
                'accuracy': accuracy,
                'precision': precision,
                'recall': recall,
                'f1': f1
            }

            cm = confusion_matrix(y_true, y_pred)
            plt.figure(figsize=(8, 6))
            sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                       xticklabels=['Negative', 'Positive'],
                       yticklabels=['Negative', 'Positive'])
            plt.title(f'Confusion Matrix - {prompt_type}')
            plt.savefig(f'confusion_matrix_{prompt_type}.png')
            plt.close()

        return pd.DataFrame(metrics).T

    def analyze_order_effects(self):
        """순서 효과 분석"""
        # 결과가 실행 순서대로 쌓여 있으므로 밈별 실행 순서 계산
        self.results_df['execution_order'] = self.results_df.groupby('image_name').cumcount() + 1

        order_accuracy = self.results_df.groupby('execution_order').apply(
            lambda x: accuracy_score(
                x['overall_sentiment'].str.lower(),
                x['predicted_sentiment']
            )
        )

        first_last_comparison = pd.DataFrame({
            'first_prompt': self.results_df[self.results_df['execution_order'] == 1]['predicted_sentiment'],
            'last_prompt': self.results_df[self.results_df['execution_order'] == 4]['predicted_sentiment']
        })

        return {
            'order_accuracy': order_accuracy,
            'first_last_comparison': first_last_comparison
        }

    def analyze_modality_interaction(self):
        """모달리티 상호작용 효과 분석"""
        accuracies = {}
        for prompt_type in ['text', 'image', 'multimodal']:
            subset = self.results_df[self.results_df['prompt_type'] == prompt_type]
            accuracies[prompt_type] = accuracy_score(
                subset['overall_sentiment'].str.lower(),
                subset['predicted_sentiment']
            )

        integration_effect = accuracies['multimodal'] - (accuracies['text'] + accuracies['image'])/2

        # 같은 밈끼리 맞춰 프롬프트 유형별 예측 비교 (라벨을 읽지 못한 응답이 있는 밈은 제외)
        modality_agreement = (
            self.results_df[self.results_df['prompt_type'].isin(['text', 'image', 'multimodal'])]
            .pivot(index='image_name', columns='prompt_type', values='predicted_sentiment')
            .dropna()[['text', 'image', 'multimodal']]
            .add_suffix('_pred')
        )

        return {
            'modal_accuracies': accuracies,
            'integration_effect': integration_effect,
            'modality_agreement': modality_agreement
        }

    def analyze_context_influence(self):
        """문맥 정보 영향력 분석"""
        baseline_subset = self.results_df[self.results_df['prompt_type'] == 'multimodal']
        context_subset = self.results_df[self.results_df['prompt_type'] == 'context_multimodal']

        baseline_acc = accuracy_score(
            baseline_subset['overall_sentiment'].str.lower(),
            baseline_subset['predicted_sentiment']
        )
        context_acc = accuracy_score(
            context_subset['overall_sentiment'].str.lower(),
            context_subset['predicted_sentiment']
        )

        # 같은 밈끼리 맞춰 기본 프롬프트와 문맥 프롬프트 예측 비교
        correction_analysis = pd.DataFrame({
            'baseline_pred': baseline_subset.set_index('image_name')['predicted_sentiment'],
            'context_pred': context_subset.set_index('image_name')['predicted_sentiment'],
            'true_label': baseline_subset.set_index('image_name')['overall_sentiment'].str.lower()
        }).dropna()

        correction_analysis['corrected'] = (
            (correction_analysis['baseline_pred'] != correction_analysis['true_label']) &
            (correction_analysis['context_pred'] == correction_analysis['true_label'])
        )

        return {
            'baseline_accuracy': baseline_acc,
            'context_accuracy': context_acc,
            'correction_rate': correction_analysis['corrected'].mean(),
            'correction_analysis': correction_analysis
        }

def run_evaluation(results_df, labels_df):
    evaluator = PerformanceEvaluator(results_df, labels_df)

    basic_metrics = evaluator.calculate_basic_metrics()
    print("\n=== 기본 성능 지표 ===")
    print(basic_metrics)

    order_effects = evaluator.analyze_order_effects()
    print("\n=== 순서 효과 분석 ===")
    print("순서별 정확도:", order_effects['order_accuracy'])

    modality_effects = evaluator.analyze_modality_interaction()
    print("\n=== 모달리티 상호작용 분석 ===")
    print("모달리티별 정확도:", modality_effects['modal_accuracies'])
    print("통합 효과:", modality_effects['integration_effect'])

    context_effects = evaluator.analyze_context_influence()
    print("\n=== 문맥 정보 영향력 분석 ===")
    print("기본 정확도:", context_effects['baseline_accuracy'])
    print("문맥 강화 정확도:", context_effects['context_accuracy'])
    print("오분류 교정 비율:", context_effects['correction_rate'])

    with open('evaluation_results.txt', 'w') as f:
        f.write("=== 성능 평가 결과 ===\n")
        f.write("\n1. 기본 성능 지표\n")
        f.write(basic_metrics.to_string())
        f.write("\n\n2. 순서 효과\n")
        f.write(str(order_effects['order_accuracy']))
        f.write("\n\n3. 모달리티 상호작용\n")
        f.write(str(modality_effects))
        f.write("\n\n4. 문맥 영향력\n")
        f.write(str(context_effects))

    return {
        'basic_metrics': basic_metrics,
        'order_effects': order_effects,
        'modality_effects': modality_effects,
        'context_effects': context_effects
    }

In [ ]:
# 저장해 둔 응답으로 평가 (API 재호출 없음)
results = pd.read_csv('meme_analysis_results.csv')

evaluation_results = run_evaluation(results, labels_df)

라벨을 읽지 못한 응답 1개 평가에서 제외



=== 기본 성능 지표 ===
                    accuracy  precision  recall        f1
context_multimodal  0.875000   1.000000    0.75  0.857143
text                0.750000   0.666667    1.00  0.800000
image               0.625000   0.571429    1.00  0.727273
multimodal          0.857143   0.750000    1.00  0.857143

=== 순서 효과 분석 ===
순서별 정확도: execution_order
1    0.750000
2    0.625000
3    0.875000
4    0.857143
dtype: float64

=== 모달리티 상호작용 분석 ===
모달리티별 정확도: {'text': 0.75, 'image': 0.625, 'multimodal': 0.8571428571428571}
통합 효과: 0.1696428571428571

=== 문맥 정보 영향력 분석 ===
기본 정확도: 0.8571428571428571
문맥 강화 정확도: 0.875
오분류 교정 비율: 0.14285714285714285
